# 🚗 Estimation du Prix des Véhicules d'Occasion — Car Dekho
**Étudiant :** Meimoune Baba Cheikh Sidiya  
**Matricule :** C34620  
**Professeur :** Ezyn SEGNANE  
**Type de tâche :** Régression supervisée  
**Dataset :** Vehicle Dataset (CarDekho) — multi-villes fusionné


## 1. Imports et Configuration

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("teal")
print("Imports OK ✅")


## 2. Chargement et Exploration des Données (EDA)

In [ ]:

# Chargement du dataset nettoyé (fusion multi-villes déjà effectuée)
df_raw = pd.read_csv('../data/processed/car_dekho_cleaned.csv')

# Correction des colonnes booléennes
bool_cols = ['fuel_Diesel','fuel_Electric','fuel_LPG','fuel_Petrol',
             'seller_type_Individual','seller_type_Trustmark Dealer','transmission_Manual']
for c in bool_cols:
    df_raw[c] = df_raw[c].astype(int)

print("Shape:", df_raw.shape)
print("\nColonnes:", df_raw.columns.tolist())
df_raw.head()


In [ ]:

# Statistiques descriptives
df_raw.describe()


In [ ]:

# Valeurs manquantes
print("Valeurs manquantes:")
print(df_raw.isnull().sum())
print("\nTypes des colonnes:")
print(df_raw.dtypes)


## 3. Analyse Exploratoire (EDA) — Visualisations

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution des variables numériques', fontsize=16, fontweight='bold')

# Prix de vente (variable cible)
axes[0,0].hist(df_raw['selling_price'], bins=50, color='#0D9488', edgecolor='white')
axes[0,0].set_title('Distribution du Prix de Vente')
axes[0,0].set_xlabel('Prix (₹)')

# Kilomètres parcourus
axes[0,1].hist(df_raw['km_driven'], bins=50, color='#3B82F6', edgecolor='white')
axes[0,1].set_title('Distribution Kilométrage')
axes[0,1].set_xlabel('km_driven (standardisé)')

# Année
axes[0,2].hist(df_raw['year'], bins=30, color='#8B5CF6', edgecolor='white')
axes[0,2].set_title("Distribution Année (standardisée)")
axes[0,2].set_xlabel('year (standardisé)')

# Âge du véhicule
axes[1,0].hist(df_raw['car_age'], bins=30, color='#F59E0B', edgecolor='white')
axes[1,0].set_title("Âge du Véhicule")
axes[1,0].set_xlabel('car_age (standardisé)')

# Prix par transmission
trans = ['Manuel', 'Automatique']
prices_trans = [
    df_raw[df_raw['transmission_Manual']==1]['selling_price'],
    df_raw[df_raw['transmission_Manual']==0]['selling_price']
]
axes[1,1].boxplot(prices_trans, labels=trans, patch_artist=True)
axes[1,1].set_title('Prix par Type de Transmission')
axes[1,1].set_ylabel('Prix (₹)')

# Prix par type de carburant
fuel_types = ['Petrol', 'Diesel', 'Electric', 'LPG', 'CNG']
fuel_prices = []
for f in fuel_types:
    col = f'fuel_{f}'
    if col in df_raw.columns:
        fuel_prices.append(df_raw[df_raw[col]==1]['selling_price'])
    else:
        fuel_prices.append(df_raw[(df_raw['fuel_Petrol']==0) & (df_raw['fuel_Diesel']==0) &
                                   (df_raw['fuel_Electric']==0) & (df_raw['fuel_LPG']==0)]['selling_price'])
axes[1,2].boxplot(fuel_prices, labels=fuel_types, patch_artist=True)
axes[1,2].set_title('Prix par Type de Carburant')
axes[1,2].set_ylabel('Prix (₹)')

plt.tight_layout()
plt.savefig('../reports/figures/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée ✅")


In [ ]:

# Matrice de corrélation
plt.figure(figsize=(12, 8))
corr = df_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0, linewidths=0.5,
            cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de Corrélation — Variables du Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Corrélation la plus forte avec selling_price:")
print(corr['selling_price'].sort_values(ascending=False))


In [ ]:

# Scatter plots : prix vs variables clés
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Relations entre variables et prix de vente', fontsize=14, fontweight='bold')

axes[0].scatter(df_raw['year'], df_raw['selling_price'], alpha=0.3, color='#0D9488', s=10)
axes[0].set_xlabel('Année (standardisée)'); axes[0].set_ylabel('Prix (₹)')
axes[0].set_title('Année vs Prix')

axes[1].scatter(df_raw['km_driven'], df_raw['selling_price'], alpha=0.3, color='#3B82F6', s=10)
axes[1].set_xlabel('Kilométrage (standardisé)'); axes[1].set_ylabel('Prix (₹)')
axes[1].set_title('Kilométrage vs Prix')

axes[2].scatter(df_raw['brand_encoded'], df_raw['selling_price'], alpha=0.3, color='#F59E0B', s=10)
axes[2].set_xlabel('Marque (encodée)'); axes[2].set_ylabel('Prix (₹)')
axes[2].set_title('Marque vs Prix')

plt.tight_layout()
plt.savefig('../reports/figures/scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()
